In [ ]:
#| default_exp handlers.general

# General Handler

`encode(yaml_path)` — marisco 2.0 のグランドマスター関数。
YAML パス 1 つを渡すだけで、全 3 フェーズ（loader → transformer → writer）を直線的に駆動し、
生 CSV/TSV を MARIS 正典 NetCDF4 へ自動整流・製本する。

In [ ]:
#| export
from __future__ import annotations
from pathlib import Path
from marisco.callbacks.core import PipelineState, run_pipeline
from marisco.handlers.pipeline.loader  import HandlerConfig, PluginSpec, load_data, gap_check
from marisco.handlers.pipeline.writer  import write_netcdf


## encode

| フェーズ | 呼び出し | 役割 |
|--------|---------|------|
| Phase 0 | `HandlerConfig.from_yaml` + `gap_check` | 型安全バインド ＆ 必須列の事前検疫 |
| Phase 1 | `load_data` | 生 CSV/TSV → `{grp: DataFrame}` |
| Phase 2 | `_load_plugin(pre_cbs)` + `build_chain` + `_load_plugin(post_cbs)` | プラグイン前段 + 11本 Soft CB + プラグイン後段（if文ゼロ） |
| Phase 3 | `write_netcdf` | 2フェーズ Strict ガード → `.nc` 物理書き出し |

`pre_cbs` / `post_cbs` が空リスト（YAML 省略時のデフォルト）の場合、アンパック展開で透過 no-op。

In [ ]:
#| export
def resolve_callback(spec: PluginSpec, yaml_dir: Path = None):
    "Return the CB *class* (not yet instantiated) for the given PluginSpec."
    return spec.resolve(yaml_dir)


def _load_plugin(spec: PluginSpec, yaml_dir: Path = None):
    "Fail-Fast dynamic import: resolve class then instantiate with spec.args."
    cls = resolve_callback(spec, yaml_dir=yaml_dir)
    return cls(**spec.args)


def _load_plugin_fn(spec: PluginSpec, yaml_dir: Path = None):
    "Fail-Fast dynamic import of a plain function (not instantiated); for custom loaders."
    return spec.resolve_fn(yaml_dir)


def _call_loader(cfg: HandlerConfig, yaml_dir: Path = None):
    "Resolve the configured loader and execute it with cfg plus any declared loader args."
    if not cfg.loader:
        return load_data(cfg)
    loader_fn = _load_plugin_fn(cfg.loader, yaml_dir=yaml_dir)
    return loader_fn(cfg, **cfg.loader.args)


def _build_normalize_cbs(cfg: HandlerConfig) -> list:
    "Assemble the normalization callbacks declared in normalize_case."
    from marisco.callbacks import LowerStripNameCB
    return [LowerStripNameCB(col_src=s, col_dst=d) for s, d in cfg.normalize_case.items()]


def _load_cfg(yaml_path: str | Path, fname_out: str = None):
    "Bind YAML to HandlerConfig and apply an optional output override."
    yaml_path = Path(yaml_path)
    yaml_dir = yaml_path.parent
    cfg = HandlerConfig.from_yaml(yaml_path)
    gap_check(cfg)
    if fname_out:
        cfg = cfg.model_copy(update={"fname_out": fname_out})
    return cfg, yaml_dir


def _init_state(cfg: HandlerConfig, yaml_dir: Path = None) -> PipelineState:
    "Load provider data and wrap it in PipelineState."
    return PipelineState(dfs=_call_loader(cfg, yaml_dir=yaml_dir))


def build_phase_pipelines(cfg: HandlerConfig) -> tuple[list, list]:
    "Split the standard core chain into pre-lossy and post-lossy phases for deep Gate 2."
    from marisco.callbacks import (
        RenameColsCB, SoftParseDateTimeCB, SoftMeltWideNuclidesCB,
        SoftConvertUnitCB, SoftRemapCB, AddSampleIDCB,
        _GuardedEncodeTimeCB, _GuardedSanitizeLonLatCB,
    )

    merged = cfg.model_copy(update={
        "rename": {**cfg.columns, **cfg.rename},
        "dt_format": cfg.time_format or cfg.dt_format,
    })
    col_provider = next((v for v in merged.rename.values() if v.endswith('_PROVIDER')), None)

    pre_lossy = [
        RenameColsCB(mapping=merged.rename, string_cast=merged.string_cast),
        SoftParseDateTimeCB(col_date=merged.col_date, col_time=merged.col_time, fmt=merged.dt_format),
        SoftMeltWideNuclidesCB(spec=[s.model_dump() for s in merged.melt_spec]),
        *[SoftConvertUnitCB(rule=r.model_dump()) for r in merged.unit_conversions],
        SoftRemapCB(col_src='NUCLIDE', col_remap='NUCLIDE', lut=merged.nuclide_lut),
        SoftRemapCB(col_src='UNIT',    col_remap='UNIT',    lut=merged.unit_lut),
        SoftRemapCB(col_src='LAB',     col_remap='LAB',     lut=merged.lab_lut),
        SoftRemapCB(col_src='NUCLIDE', col_remap='AREA',    lut={}, default_val=merged.area_default),
    ]
    post_lossy = [
        _GuardedSanitizeLonLatCB(),
        _GuardedEncodeTimeCB(),
        AddSampleIDCB(col_provider=col_provider),
    ]
    return pre_lossy, post_lossy


def _run_preflight(state: PipelineState, cfg: HandlerConfig, yaml_dir: Path = None) -> PipelineState:
    "Run all declarative, non-lossy transformations and Gate 2 in memory."
    pre_lossy, _ = build_phase_pipelines(cfg)
    chain = [
        *_build_normalize_cbs(cfg),
        *[_load_plugin(s, yaml_dir=yaml_dir) for s in cfg.pre_cbs],
        *pre_lossy,
    ]
    run_pipeline(state, chain)
    gap_check(cfg, state.dfs)
    return state


def _run_finalize(state: PipelineState, cfg: HandlerConfig, yaml_dir: Path = None) -> PipelineState:
    "Run the lossy/output-facing phase after preflight has passed."
    _, post_lossy = build_phase_pipelines(cfg)
    chain = [
        *post_lossy,
        *[_load_plugin(s, yaml_dir=yaml_dir) for s in cfg.post_cbs],
    ]
    run_pipeline(state, chain)
    return state


def _verify_success_message(cfg: HandlerConfig, state: PipelineState) -> str:
    lines = [
        f"Congratulations - Gate 1 and Gate 2 both passed for {cfg.title or cfg.module_name!r}.",
        "The declarative pipeline is physically consistent through the pre-lossy checkpoint.",
    ]
    for grp, df in state.dfs.items():
        lines.append(f"- {grp}: {len(df):,} rows x {df.shape[1]} cols after declarative canonicalization")
    return "
".join(lines)


def build_core_pipeline(cfg: HandlerConfig) -> list:
    "Auto-assemble the standard core CB chain with topology guards."
    pre_lossy, post_lossy = build_phase_pipelines(cfg)
    return [*pre_lossy, *post_lossy]


def verify(yaml_path: str | Path, fname_out: str = None) -> PipelineState:
    "Dry-run a YAML-configured dataset through Gate 1, declarative transforms, and deep Gate 2 without writing NetCDF."
    cfg, yaml_dir = _load_cfg(yaml_path, fname_out=fname_out)
    state = _init_state(cfg, yaml_dir=yaml_dir)
    _run_preflight(state, cfg, yaml_dir=yaml_dir)
    print(_verify_success_message(cfg, state))
    return state


def encode(yaml_path: str | Path, fname_out: str = None) -> None:
    "Encode any YAML-configured dataset to MARIS NetCDF4 in a pure, phase-aware pipeline."
    cfg, yaml_dir = _load_cfg(yaml_path, fname_out=fname_out)
    state = _init_state(cfg, yaml_dir=yaml_dir)
    _run_preflight(state, cfg, yaml_dir=yaml_dir)
    _run_finalize(state, cfg, yaml_dir=yaml_dir)
    write_netcdf(state, cfg)


In [ ]:
import sys
sys.stdout.reconfigure(encoding="utf-8")
from pathlib import Path
from marisco.handlers.pipeline.loader import HandlerConfig, PluginSpec, gap_check
from marisco.handlers.general import encode, _load_plugin, build_core_pipeline, resolve_callback

cfg = HandlerConfig.from_yaml("config/handlers/fram_strait.yaml")
gap_check(cfg)

core = build_core_pipeline(cfg)
print(f"build_core_pipeline (FramStrait, 1 unit_conv): {len(core)} CBs")
for i, cb in enumerate(core, 1):
    print(f"  {i:2d}. {type(cb).__name__}")

# Topology guard smoke test: empty dfs → guards degrade to Null-Object
from marisco.callbacks import Transformer
import pandas as pd
tfm_empty = Transformer({'SEAWATER': pd.DataFrame()}, cbs=core)
tfm_empty()  # must not raise KeyError for TIME/LON/LAT absent
print("\nTopology guard ✓ — no KeyError on empty DataFrame (Null-Object auto-degrade)")

# ── resolve_callback: legacy path ──
spec_path = PluginSpec(path="marisco.callbacks.shared.SoftRegexTransformCB")
cls = resolve_callback(spec_path)
from marisco.callbacks.shared import SoftRegexTransformCB
assert cls is SoftRegexTransformCB, "legacy path resolution failed"
print("resolve_callback(path) ✓")

# ── resolve_callback: name shorthand (shared scan) ──
spec_name = PluginSpec(name="SoftDMStoDecimalCB")
cls = resolve_callback(spec_name)
from marisco.callbacks.shared import SoftDMStoDecimalCB
assert cls is SoftDMStoDecimalCB, "name shorthand resolution failed"
print("resolve_callback(name) ✓")

# ── resolve_callback: local file ──
import tempfile, textwrap
dummy_src = textwrap.dedent("""
    from marisco.callbacks.core import PerGroupCB
    class DummyLocalCB(PerGroupCB):
        grps = ['SEAWATER']
        def each_grp(self, grp, df, tfm): pass
""")
with tempfile.TemporaryDirectory() as tmpdir:
    local_py = Path(tmpdir) / "local_cbs.py"
    local_py.write_text(dummy_src)
    spec_file = PluginSpec(**{"file": "local_cbs.py", "class": "DummyLocalCB"})
    cls = resolve_callback(spec_file, yaml_dir=Path(tmpdir))
    assert cls.__name__ == "DummyLocalCB"
    instance = _load_plugin(spec_file, yaml_dir=Path(tmpdir))
    assert type(instance).__name__ == "DummyLocalCB"
print("resolve_callback(file+class) ✓")

# ── Fail-Fast: unknown name raises ImportError ──
try:
    resolve_callback(PluginSpec(name="NonExistentCB"))
    assert False, "Should have raised"
except ImportError:
    pass
print("Fail-Fast unknown name ✓")

# ── Fail-Fast: bad path raises immediately ──
try:
    _load_plugin(PluginSpec(path="marisco.nonexistent.FakeCB"))
except (ImportError, ModuleNotFoundError) as e:
    print(f"Fail-Fast bad path ✓ — {type(e).__name__}")

# normalize_case smoke test
cfg_norm = cfg.model_copy(update={"normalize_case": {"nuclide_raw": "NUCLIDE"}})
from marisco.callbacks import LowerStripNameCB
norm_cbs = [LowerStripNameCB(col_src=s, col_dst=d) for s, d in cfg_norm.normalize_case.items()]
assert len(norm_cbs) == 1 and type(norm_cbs[0]).__name__ == "LowerStripNameCB"
print("normalize_case → LowerStripNameCB ✓")

In [ ]:
#|eval: false
# Full pipeline integration test (requires network access to Zenodo)
encode("config/handlers/fram_strait.yaml", fname_out="_data/output/FramStrait_general.nc")
print("FramStrait_general.nc written.")